In [9]:
import sqlite3 
import pandas as pd

## Importing raw observations

In [10]:
conn = sqlite3.connect('../data/probes.db')
df_raw_observations = pd.read_sql_query("SELECT * FROM probes", conn)

## Cleaning

- During collection, the WiFi adapter failed at times, resulting in incomplete scan cycles.
- On average every scan cycle lasted 30-40 seconds (10 seconds for channel 1,6,11 + changing the channel latency), thus all cycles that are under 30 seconds are faulty cycles
- Since occupancy does not change drastically every 30 seconds, I decided to impute the faulty cycles with the previous/subsequent cycle (if the date matches)

In [11]:
df_scan = pd.read_sql_query("""
    SELECT 
        scan_cycle_id,
        timestamp,
        DATE(timestamp, 'unixepoch') AS date,
        ROUND(MAX(timestamp) - MIN(timestamp), 2) AS total_duration_seconds,
        COUNT(*) AS total_probes,
        COUNT(CASE WHEN rssi >= -55 THEN 1 END) AS probes_rssi_55, 
        COUNT(CASE WHEN rssi >= -60 THEN 1 END) AS probes_rssi_60, 
        COUNT(CASE WHEN rssi >= -65 THEN 1 END) AS probes_rssi_65, 
        COUNT(CASE WHEN rssi >= -70 THEN 1 END) AS probes_rssi_70, 
        COUNT(CASE WHEN rssi >= -75 THEN 1 END) AS probes_rssi_75,
        COUNT(CASE WHEN rssi >= -80 THEN 1 END) AS probes_rssi_80
    FROM probes 
    GROUP BY scan_cycle_id 
    ORDER BY scan_cycle_id
""", conn)

df_scan.head()

,scan_cycle_id,timestamp,date,total_duration_seconds,total_probes,probes_rssi_55,probes_rssi_60,probes_rssi_65,probes_rssi_70,probes_rssi_75,probes_rssi_80
0,1,1.787343e+09,2026-08-21,34.66,117,13,19,34,47,75,98
1,2,1.787343e+09,2026-08-21,35.78,110,5,19,31,54,79,99
2,3,1.787343e+09,2026-08-21,35.16,100,4,22,42,60,76,82
3,4,1.787343e+09,2026-08-21,35.58,67,5,7,18,28,53,63
4,5,1.787343e+09,2026-08-21,33.82,158,28,33,53,98,129,141


In [12]:
faulty_cycles = []
replacement = []
feature_columns = [col for col in df_scan if col not in ['scan_cycle_id', 'timestamp', 'date']]

for x in df_scan.index:
    prev_valid = (x > df_scan.index[0] and 
                     df_scan.loc[x, 'date'] == df_scan.loc[x-1, 'date'] and 
                     df_scan.loc[x-1, 'total_duration_seconds'] > 30)
    next_valid = (x < df_scan.index[-1] and 
              df_scan.loc[x, 'date'] == df_scan.loc[x+1, 'date'] and 
              df_scan.loc[x+1, 'total_duration_seconds'] > 30)
    if df_scan.loc[x, 'total_duration_seconds'] < 30:
        if prev_valid:
          faulty_cycles.append(x)
          replacement.append(x - 1)
          df_scan.loc[x, feature_columns] = df_scan.loc[x - 1, feature_columns]
        elif next_valid:
          faulty_cycles.append(x)
          replacement.append(x + 1)
          df_scan.loc[x, feature_columns] = df_scan.loc[x + 1, feature_columns]

print(f"Faulty cycles: {faulty_cycles}, Amount: {len(faulty_cycles)}")
df_scan.tail()

Faulty cycles: [103, 111, 179, 204, 366, 399, 513, 700, 705, 732, 844, 899, 910, 1019, 1065, 1074, 1113, 1449], Amount: 18


,scan_cycle_id,timestamp,date,total_duration_seconds,total_probes,probes_rssi_55,probes_rssi_60,probes_rssi_65,probes_rssi_70,probes_rssi_75,probes_rssi_80
1445,1446,1.788459e+09,2026-09-03,37.87,324,22,52,126,183,232,253
1446,1447,1.788459e+09,2026-09-03,37.93,385,27,29,41,104,209,292
1447,1448,1.788459e+09,2026-09-03,38.12,304,7,23,79,190,261,288
1448,1449,1.788459e+09,2026-09-03,39.42,320,4,12,48,137,253,298
1449,1450,1.788459e+09,2026-09-03,39.42,320,4,12,48,137,253,298


## Summary data frame

- Occupancy during the data collection occupancy was calculated every 5 minutes, thus the same rule will be follwed for the prediction model
- The 1450 rows of 30 second scan cycles, were summarized every 10 cycles to make 5 minute windows
- MAC address' metrics were calculated directly from ``` df_raw_observations``` since it is intended to calculate distinct addresses within every 5 minute cycle

In [ ]:
# No need to normalize or drop rows because it represents >2% of the data and mac count is not linear with occupancy

mac_addresses = pd.read_sql_query(
    """
    WITH mac_counts AS (
    SELECT 
        (scan_cycle_id) / 10 AS window_id,
        mac_address,
        COUNT(*) AS times_seen
    FROM probes
    GROUP BY window_id, mac_address
    )
    SELECT 
        window_id,
        COUNT(mac_address) AS unique_mac_addresses,
        COUNT(CASE WHEN times_seen >= 2 THEN 1 END) AS unique_mac_2plus,
        COUNT(CASE WHEN times_seen >= 3 THEN 1 END) AS unique_mac_3plus,
        COUNT(CASE WHEN times_seen >= 4 THEN 1 END) AS unique_mac_4plus,
        COUNT(CASE WHEN times_seen >= 5 THEN 1 END) AS unique_mac_5plus
    FROM mac_counts
    GROUP BY window_id
    ORDER BY window_id
    """,
    conn,
)

df_5min = df_scan.groupby((df_scan["scan_cycle_id"] - 1) // 10 + 1).agg(
    timestamp=("timestamp", "first"),
    total_probes=("total_probes", "sum"),
    probes_rssi_55=("probes_rssi_55", "sum"),
    probes_rssi_60=("probes_rssi_60", "sum"),
    probes_rssi_65=("probes_rssi_65", "sum"),
    probes_rssi_70=("probes_rssi_70", "sum"),
    probes_rssi_75=("probes_rssi_75", "sum"),
)
mac_addresses.set_index("window_id", inplace=True)
df = pd.merge(df_5min, mac_addresses, left_index=True, right_index=True)
df["ground_truth_people"] = [
    22,
    29,
    31,
    26,
    28,
    27,
    37,
    25,
    25,
    37,
    28,
    27,
    25,
    26,
    33,
    45,
    34,
    28,
    6,
    6,
    4,
    5,
    6,
    4,
    5,
    7,
    10,
    12,
    13,
    15,
    16,
    17,
    17,
    14,
    16,
    8,
    5,
    7,
    6,
    7,
    5,
    6,
    19,
    22,
    20,
    22,
    25,
    28,
    29,
    29,
    32,
    34,
    35,
    38,
    43,
    48,
    50,
    42,
    42,
    43,
    48,
    49,
    52,
    52,
    52,
    52,
    43,
    45,
    54,
    52,
    8,
    8,
    10,
    12,
    12,
    12,
    9,
    10,
    8,
    8,
    8,
    8,
    10,
    11,
    11,
    12,
    10,
    6,
    7,
    6,
    7,
    8,
    8,
    6,
    7,
    6,
    6,
    8,
    8,
    11,
    9,
    9,
    8,
    8,
    10,
    10,
    10,
    6,
    4,
    4,
    7,
    6,
    8,
    112,
    119,
    106,
    105,
    125,
    108,
    113,
    111,
    90,
    86,
    72,
    61,
    46,
    45,
    47,
    46,
    50,
    48,
    39,
    59,
    74,
    101,
    124,
    144,
    165,
    162,
    184,
    251,
    272,
    281,
    302,
    318,
]
print(len(df))
df.head()

145


,timestamp,total_probes,probes_rssi_55,probes_rssi_60,probes_rssi_65,probes_rssi_70,probes_rssi_75,unique_mac_addresses,unique_mac_2plus,unique_mac_3plus,unique_mac_4plus,unique_mac_5plus,ground_truth_people
scan_cycle_id,,,,,,,,,,,,,
1,1.787343e+09,1192,95,192,380,615,863,608,168,51,26,24,22
2,1.787343e+09,1083,42,143,297,554,852,653,187,45,26,15,29
3,1.787344e+09,1200,133,234,404,643,925,523,155,49,31,18,31
4,1.787344e+09,924,23,70,218,456,649,470,120,36,27,21,26
5,1.787345e+09,966,50,107,204,413,701,600,174,43,23,17,28
